# 06 Stage 1 — Full Model Comparison

This notebook compares **four** trained classifiers across **three** barrier targets (**12 evaluations**):

- Models: Logistic Regression, Decision Tree, Random Forest, XGBoost
- Targets: Household, Logistic, Facility

**Rules:** Step 4 loads from `saved_models/stage1/`. If `.pkl` files are absent (typical after git clone), set `TRAIN_PICKLES_IF_MISSING = True` (default) to fit all 12 models once via `src.models.stage1_pickles.build_all_stage1_models` using the **same** `split_and_scale` pipeline, then load them.  
Evaluation still uses **only** held-out test folds — no leakage.


In [1]:
# Step 1 — Imports & project root (for src.* and stable output paths)
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
    confusion_matrix,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.split_scale import split_and_scale
from src.evaluation.metrics import evaluate_model


In [2]:
# Step 2 — Load processed features and targets; print shapes & class distributions
processed_dir = PROJECT_ROOT / "data" / "processed"

X = pd.read_csv(processed_dir / "X_features.csv")

y_household = pd.read_csv(processed_dir / "y_household.csv").squeeze("columns")
y_logistic = pd.read_csv(processed_dir / "y_logistic.csv").squeeze("columns")
y_facility = pd.read_csv(processed_dir / "y_facility.csv").squeeze("columns")

y_map = {
    "household": y_household,
    "logistic": y_logistic,
    "facility": y_facility,
}

print("Features shape:", X.shape)
for key, ys in y_map.items():
    print(
        f"Target {key}: len={len(ys)} | value_counts:\n"
        f"{ys.value_counts().sort_index()}\n"
    )


Features shape: (724115, 56)
Target household: len=724115 | value_counts:
target_household
0    527477
1    196638
Name: count, dtype: int64

Target logistic: len=724115 | value_counts:
target_logistic
0    495248
1    228867
Name: count, dtype: int64

Target facility: len=724115 | value_counts:
target_facility
0    390968
1    333147
Name: count, dtype: int64



In [3]:
# Step 3 — Train/test splits (stratified) + scaling via project helper

TARGET_CONFIG = [
    ("household", "target_household", "Household Barrier"),
    ("logistic", "target_logistic", "Logistic Barrier"),
    ("facility", "target_facility", "Facility Barrier"),
]

splits = {}
for key, target_col, _ in TARGET_CONFIG:
    combo = pd.concat([X, y_map[key].rename(target_col)], axis=1)
    X_train, X_test, y_train, y_test, scaler = split_and_scale(combo, target_col, apply_scaling=True)
    splits[key] = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "scaler": scaler,
        "target_col": target_col,
    }
    X_test_np = np.asarray(X_test)
    print(f"Splits ready for '{key}': StandardScaler fitted on train · test matrix shape {X_test_np.shape}")


target_household - Train: 579292 rows | Test: 144823 rows


Splits ready for 'household': StandardScaler fitted on train · test matrix shape (144823, 56)


target_logistic - Train: 579292 rows | Test: 144823 rows


Splits ready for 'logistic': StandardScaler fitted on train · test matrix shape (144823, 56)


target_facility - Train: 579292 rows | Test: 144823 rows


Splits ready for 'facility': StandardScaler fitted on train · test matrix shape (144823, 56)


In [4]:
# Step 4 — Load all 12 trained estimators from disk (no retraining)
#
# .pkl files are gitignored — clone alone will not contain them.

import os

_env_dir = os.environ.get("BARRIERLENS_STAGE1_MODEL_DIR")
if _env_dir:
    MODEL_DIR = Path(_env_dir).expanduser().resolve()
else:
    MODEL_DIR = (PROJECT_ROOT / "saved_models" / "stage1").resolve()

MODEL_DIR.mkdir(parents=True, exist_ok=True)
print("Model directory:", MODEL_DIR)

_suffixes = ("household", "logistic", "facility")
_prefixes = ("logistic_regression", "decision_tree", "random_forest", "xgboost")
_required = tuple(f"{p}_{t}.pkl" for p in _prefixes for t in _suffixes)
_missing = [fname for fname in _required if not (MODEL_DIR / fname).is_file()]
TRAIN_PICKLES_IF_MISSING = True  # set False to only load (fail if .pkl not present)
if _missing:
    if not TRAIN_PICKLES_IF_MISSING:
        hint_lines = [
            "Stage-1 pickle files are missing (.gitignore excludes them — not in git clone).",
            "Expected all 12 files under:",
            f"    {MODEL_DIR}",
            "",
            "Generate locally, for example:",
            "    - notebooks/02_stage1_logistic.ipynb → logistic_regression_<target>.pkl",
            "    - notebooks/04_stage1_random_forest.ipynb → random_forest_<target>.pkl",
            "    - Or set TRAIN_PICKLES_IF_MISSING = True (default) to train via src.models.stage1_pickles.",
            "",
            "If pickles live elsewhere: copy into saved_models/stage1/",
            'or set os.environ["BARRIERLENS_STAGE1_MODEL_DIR"]=r"C:\\path\\to\\folder"',
            "",
            f"Missing {len(_missing)} file(s):",
            *[f"  - {m}" for m in _missing],
        ]
        raise FileNotFoundError("\n".join(hint_lines))
    print(
        "Missing .pkl files — training all 12 models using split_and_scale (may take a few minutes)..."
    )
    from src.models.stage1_pickles import build_all_stage1_models

    build_all_stage1_models(PROJECT_ROOT)
    _missing = [fname for fname in _required if not (MODEL_DIR / fname).is_file()]
    if _missing:
        raise RuntimeError(
            "Still missing after build_all_stage1_models:\n" + "\n".join(_missing)
        )


def load_pkl(stem: str):
    return joblib.load(MODEL_DIR / f"{stem}.pkl")


def load_bundle():
    # MODELS[display_model_name][target_key] -> fitted estimator
    logistic, dtree, rforest, xgbm = {}, {}, {}, {}

    for key in _suffixes:
        logistic[key] = load_pkl(f"logistic_regression_{key}")
        dtree[key] = load_pkl(f"decision_tree_{key}")
        rforest[key] = load_pkl(f"random_forest_{key}")
        xgbm[key] = load_pkl(f"xgboost_{key}")

    return {
        "Logistic Regression": logistic,
        "Decision Tree": dtree,
        "Random Forest": rforest,
        "XGBoost": xgbm,
    }


MODELS = load_bundle()
print("Loaded 12 pickled models.")


Model directory: C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\saved_models\stage1


Loaded 12 pickled models.


In [5]:
# Step 5–6 — evaluate_model() on held-out scaled test features (12 runs)

DISPLAY_ORDER = ["Logistic Regression", "Decision Tree", "Random Forest", "XGBoost"]
results_records = []

for model_name in DISPLAY_ORDER:
    for key, _, barrier_label in TARGET_CONFIG:
        mdl = MODELS[model_name][key]
        X_test = splits[key]["X_test"]
        y_test = splits[key]["y_test"]

        metrics = evaluate_model(mdl, X_test, y_test, model_name, barrier_label)
        results_records.append(metrics)

print(f"Collected {len(results_records)} evaluation rows.")



=== Logistic Regression | Household Barrier barrier ===
  Model       : Logistic Regression
  Target      : Household Barrier
  Accuracy    : 0.7298
  ROC-AUC     : 0.6586
  Precision   : 0.5458
  Recall      : 0.0304
  F1-Score    : 0.0577
              precision    recall  f1-score   support

           0       0.73      0.99      0.84    105495
           1       0.55      0.03      0.06     39328

    accuracy                           0.73    144823
   macro avg       0.64      0.51      0.45    144823
weighted avg       0.68      0.73      0.63    144823




=== Logistic Regression | Logistic Barrier barrier ===
  Model       : Logistic Regression
  Target      : Logistic Barrier
  Accuracy    : 0.6896
  ROC-AUC     : 0.6603
  Precision   : 0.5512
  Recall      : 0.096
  F1-Score    : 0.1635
              precision    recall  f1-score   support

           0       0.70      0.96      0.81     99050
           1       0.55      0.10      0.16     45773

    accuracy                           0.69    144823
   macro avg       0.62      0.53      0.49    144823
weighted avg       0.65      0.69      0.61    144823




=== Logistic Regression | Facility Barrier barrier ===
  Model       : Logistic Regression
  Target      : Facility Barrier
  Accuracy    : 0.5819
  ROC-AUC     : 0.6112
  Precision   : 0.5582
  Recall      : 0.4372
  F1-Score    : 0.4904
              precision    recall  f1-score   support

           0       0.60      0.71      0.65     78194
           1       0.56      0.44      0.49     66629

    accuracy                           0.58    144823
   macro avg       0.58      0.57      0.57    144823
weighted avg       0.58      0.58      0.57    144823




=== Decision Tree | Household Barrier barrier ===
  Model       : Decision Tree
  Target      : Household Barrier
  Accuracy    : 0.7305
  ROC-AUC     : 0.6511
  Precision   : 0.5642
  Recall      : 0.033
  F1-Score    : 0.0623
              precision    recall  f1-score   support

           0       0.73      0.99      0.84    105495
           1       0.56      0.03      0.06     39328

    accuracy                           0.73    144823
   macro avg       0.65      0.51      0.45    144823
weighted avg       0.69      0.73      0.63    144823




=== Decision Tree | Logistic Barrier barrier ===
  Model       : Decision Tree
  Target      : Logistic Barrier
  Accuracy    : 0.6897
  ROC-AUC     : 0.6566
  Precision   : 0.5658
  Recall      : 0.0779
  F1-Score    : 0.137
              precision    recall  f1-score   support

           0       0.70      0.97      0.81     99050
           1       0.57      0.08      0.14     45773

    accuracy                           0.69    144823
   macro avg       0.63      0.53      0.47    144823
weighted avg       0.65      0.69      0.60    144823



...


=== Random Forest | Logistic Barrier barrier ===
  Model       : Random Forest
  Target      : Logistic Barrier
  Accuracy    : 0.6029
  ROC-AUC     : 0.663
  Precision   : 0.4184
  Recall      : 0.6575
  F1-Score    : 0.5114
              precision    recall  f1-score   support

           0       0.78      0.58      0.67     99050
           1       0.42      0.66      0.51     45773

    accuracy                           0.60    144823
   macro avg       0.60      0.62      0.59    144823
weighted avg       0.67      0.60      0.62    144823




=== Random Forest | Facility Barrier barrier ===
  Model       : Random Forest
  Target      : Facility Barrier
  Accuracy    : 0.5811
  ROC-AUC     : 0.617
  Precision   : 0.5404
  Recall      : 0.5989
  F1-Score    : 0.5682
              precision    recall  f1-score   support

           0       0.62      0.57      0.59     78194
           1       0.54      0.60      0.57     66629

    accuracy                           0.58    144823
   macro avg       0.58      0.58      0.58    144823
weighted avg       0.59      0.58      0.58    144823




=== XGBoost | Household Barrier barrier ===
  Model       : XGBoost
  Target      : Household Barrier
  Accuracy    : 0.7307
  ROC-AUC     : 0.6625
  Precision   : 0.5486
  Recall      : 0.0464
  F1-Score    : 0.0855
              precision    recall  f1-score   support

           0       0.73      0.99      0.84    105495
           1       0.55      0.05      0.09     39328

    accuracy                           0.73    144823
   macro avg       0.64      0.52      0.46    144823
weighted avg       0.68      0.73      0.64    144823




=== XGBoost | Logistic Barrier barrier ===
  Model       : XGBoost
  Target      : Logistic Barrier
  Accuracy    : 0.6914
  ROC-AUC     : 0.665
  Precision   : 0.5638
  Recall      : 0.1037
  F1-Score    : 0.1752
              precision    recall  f1-score   support

           0       0.70      0.96      0.81     99050
           1       0.56      0.10      0.18     45773

    accuracy                           0.69    144823
   macro avg       0.63      0.53      0.49    144823
weighted avg       0.66      0.69      0.61    144823




=== XGBoost | Facility Barrier barrier ===
  Model       : XGBoost
  Target      : Facility Barrier
  Accuracy    : 0.5864
  ROC-AUC     : 0.6188
  Precision   : 0.5653
  Recall      : 0.4371
  F1-Score    : 0.493
              precision    recall  f1-score   support

           0       0.60      0.71      0.65     78194
           1       0.57      0.44      0.49     66629

    accuracy                           0.59    144823
   macro avg       0.58      0.58      0.57    144823
weighted avg       0.58      0.59      0.58    144823

Collected 12 evaluation rows.


In [6]:
# Step 7 — comparison DataFrame sorted by ROC-AUC (primary ranking)

results_df = pd.DataFrame(results_records)
column_order = ["Model", "Target", "Accuracy", "ROC-AUC", "Precision", "Recall", "F1-Score"]
results_df = results_df[column_order]
results_sorted = results_df.sort_values("ROC-AUC", ascending=False).reset_index(drop=True)

results_sorted


,Model,Target,Accuracy,ROC-AUC,Precision,Recall,F1-Score
0,XGBoost,Logistic Barrier,0.6914,0.6650,0.5638,0.1037,0.1752
1,Random Forest,Logistic Barrier,0.6029,0.6630,0.4184,0.6575,0.5114
2,XGBoost,Household Barrier,0.7307,0.6625,0.5486,0.0464,0.0855
3,Random Forest,Household Barrier,0.6039,0.6613,0.3693,0.6479,0.4704
4,Logistic Regression,Logistic Barrier,0.6896,0.6603,0.5512,0.0960,0.1635
5,Logistic Regression,Household Barrier,0.7298,0.6586,0.5458,0.0304,0.0577
6,Decision Tree,Logistic Barrier,0.6897,0.6566,0.5658,0.0779,0.1370
7,Decision Tree,Household Barrier,0.7305,0.6511,0.5642,0.0330,0.0623
8,XGBoost,Facility Barrier,0.5864,0.6188,0.5653,0.4371,0.4930
9,Random Forest,Facility Barrier,0.5811,0.6170,0.5404,0.5989,0.5682


In [7]:
# Step 8 — persist CSV summary

out_dir = PROJECT_ROOT / "outputs" / "stage1_results"
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / "model_comparison_table.csv"
results_sorted.to_csv(csv_path, index=False)
print(f"Saved comparison table -> {csv_path}")


Saved comparison table -> C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\outputs\stage1_results\model_comparison_table.csv


In [8]:
# Step 9A — ROC-AUC bar plots (one figure per barrier target)


def roc_auc_bar_per_target(save_prefix: str = "model_compare_bar_rocauc"):
    for key, _, barrier_label in TARGET_CONFIG:
        sub = results_df[results_df["Target"] == barrier_label].copy()

        plt.figure(figsize=(8.5, 5))
        ax = sns.barplot(data=sub, x="Model", y="ROC-AUC", order=DISPLAY_ORDER, palette="muted")
        ax.set_title(f"ROC-AUC by model — {barrier_label}")
        ax.set_ylim(0, 1)
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()

        fout = out_dir / f"{save_prefix}_{key}.png"
        plt.savefig(fout, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"Saved bar plot -> {fout}")


roc_auc_bar_per_target()


C:\Users\hireg\AppData\Local\Temp\ipykernel_23048\4288522431.py:9: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=sub, x="Model", y="ROC-AUC", order=DISPLAY_ORDER, palette="muted")


Saved bar plot -> C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\outputs\stage1_results\model_compare_bar_rocauc_household.png


C:\Users\hireg\AppData\Local\Temp\ipykernel_23048\4288522431.py:9: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=sub, x="Model", y="ROC-AUC", order=DISPLAY_ORDER, palette="muted")


Saved bar plot -> C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\outputs\stage1_results\model_compare_bar_rocauc_logistic.png


C:\Users\hireg\AppData\Local\Temp\ipykernel_23048\4288522431.py:9: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=sub, x="Model", y="ROC-AUC", order=DISPLAY_ORDER, palette="muted")


Saved bar plot -> C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\outputs\stage1_results\model_compare_bar_rocauc_facility.png


In [9]:
# Step 9B — combined comparison chart (ROC-AUC grouped by target)

combo_plot_path = out_dir / "model_compare_rocauc_grouped_by_target.png"

plt.figure(figsize=(11, 6))
ax = sns.barplot(
    data=results_df,
    x="Model",
    y="ROC-AUC",
    hue="Target",
    order=DISPLAY_ORDER,
    palette="Set2",
)
ax.set_title("ROC-AUC — all models × targets (held-out test set)")
ax.set_ylim(0, 1)
plt.xticks(rotation=20, ha="right")
plt.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(combo_plot_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved combined ROC-AUC chart -> {combo_plot_path}")


Saved combined ROC-AUC chart -> C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\outputs\stage1_results\model_compare_rocauc_grouped_by_target.png


In [10]:
# Step 9C — confusion matrices for each model × target (saved under confusion_matrices/model_compare)


def sanitize(name: str) -> str:
    return name.lower().replace(" ", "_")


cm_dir = out_dir / "confusion_matrices" / "model_compare"
cm_dir.mkdir(parents=True, exist_ok=True)

for model_name in DISPLAY_ORDER:
    for key, _, barrier_label in TARGET_CONFIG:
        mdl = MODELS[model_name][key]
        Xt = splits[key]["X_test"]
        yt = splits[key]["y_test"]
        pred = mdl.predict(Xt)
        cm = confusion_matrix(yt, pred)

        plt.figure(figsize=(4.8, 4))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Pred 0", "Pred 1"],
            yticklabels=["True 0", "True 1"],
        )
        plt.title(f"{model_name} | {barrier_label}")
        plt.ylabel("Actual")
        plt.xlabel("Predicted")
        plt.tight_layout()

        fname = cm_dir / f"cm_{sanitize(model_name)}_{key}.png"
        plt.savefig(fname, dpi=150, bbox_inches="tight")
        plt.close()

print(f"Saved {len(DISPLAY_ORDER) * len(TARGET_CONFIG)} confusion-matrix PNGs -> {cm_dir}")


Saved 12 confusion-matrix PNGs -> C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\outputs\stage1_results\confusion_matrices\model_compare


In [11]:
# Step 10 — ROC overlays (predict_proba), one PNG per target -> outputs/stage1_results/roc_curves/


def plot_roc_overlay_for_target(target_key: str, barrier_title: str) -> None:
    roc_dir = out_dir / "roc_curves"
    roc_dir.mkdir(parents=True, exist_ok=True)

    Xt = splits[target_key]["X_test"]
    yt = splits[target_key]["y_test"]

    plt.figure(figsize=(8.5, 6.5))
    for model_name in DISPLAY_ORDER:
        mdl = MODELS[model_name][target_key]
        y_prob = mdl.predict_proba(Xt)[:, 1]
        fpr, tpr, _ = roc_curve(yt, y_prob)
        roc_auc_val = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2.0, label=f"{model_name} (AUC={roc_auc_val:.4f})")
    plt.plot([0, 1], [0, 1], "k--", lw=1, label="Random baseline")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title(f"ROC overlay — {barrier_title}")
    plt.legend(loc="lower right")
    plt.xlim([-0.02, 1.0])
    plt.ylim([0.0, 1.02])
    plt.tight_layout()

    out_png = roc_dir / f"roc_overlay_{target_key}.png"
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved ROC overlay -> {out_png}")


for key, _, label in TARGET_CONFIG:
    plot_roc_overlay_for_target(key, label)


Saved ROC overlay -> C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\outputs\stage1_results\roc_curves\roc_overlay_household.png


Saved ROC overlay -> C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\outputs\stage1_results\roc_curves\roc_overlay_logistic.png


Saved ROC overlay -> C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\outputs\stage1_results\roc_curves\roc_overlay_facility.png


In [12]:
# Step 11 — final insights

focus = results_df.copy()

best_per_target_idx = focus.groupby("Target")["ROC-AUC"].idxmax()
best_per_target = focus.loc[best_per_target_idx, ["Model", "Target", "ROC-AUC"]].reset_index(drop=True)

overall_idx = focus["ROC-AUC"].idxmax()
overall_row = focus.loc[[overall_idx], ["Model", "Target", "ROC-AUC"]]

avg_by_model = focus.groupby("Model", as_index=False)["ROC-AUC"].mean().sort_values("ROC-AUC", ascending=False)

print("\n===== Best model per barrier (ROC-AUC) =====")
for _, row in best_per_target.iterrows():
    print(f"  {row['Target']:<22} -> {row['Model']:<20} ROC-AUC={row['ROC-AUC']}")

print("\n===== Overall best single evaluation (ROC-AUC) =====")
for _, row in overall_row.iterrows():
    print(f"  {row['Model']} | {row['Target']} -> ROC-AUC={row['ROC-AUC']}")

print("\n===== Mean ROC-AUC across barriers (ranking sanity check) =====")
print(avg_by_model.to_string(index=False))

print(
    "\nNote — expected qualitative pattern for tuned ensembles: "
    "XGBoost ≥ Random Forest > Logistic Regression ≥ Decision Tree. "
    "Compare with tables and plots above."
)



===== Best model per barrier (ROC-AUC) =====
  Facility Barrier       -> XGBoost              ROC-AUC=0.6188
  Household Barrier      -> XGBoost              ROC-AUC=0.6625
  Logistic Barrier       -> XGBoost              ROC-AUC=0.665

===== Overall best single evaluation (ROC-AUC) =====
  XGBoost | Logistic Barrier -> ROC-AUC=0.665

===== Mean ROC-AUC across barriers (ranking sanity check) =====
              Model  ROC-AUC
            XGBoost 0.648767
      Random Forest 0.647100
Logistic Regression 0.643367
      Decision Tree 0.637833

Note — expected qualitative pattern for tuned ensembles: XGBoost ≥ Random Forest > Logistic Regression ≥ Decision Tree. Compare with tables and plots above.
